# Сверка двух портфелей и факторный анализ резервов

Ноутбук объединяет два Excel-портфеля по ключу:

**УНП / номер договора / Актив(УО) / валюта**

и строит факторный анализ изменения резервов.

### Основные факторы

- **Расходы на резервы** = `Резерв_1 - Резерв_2`
- **Изменение качества** = `0`, если `ГР_1 = ГР_2`, иначе  
  `-Задолженность_2 × (%рез_2 - %рез_1) / 100`
- **Переоценка** =  
  `-Задолженность_1_в_валюте × (Курс_2 - Курс_1) × %рез_1 / 100`
- **Движение портфеля** =  
  `-(Задолженность_2 × %рез_1 / 100 - Задолженность_1 × %рез_1 / 100) - Переоценка`

Для договоров, которые есть только во втором портфеле:

- изменение качества = `0`;
- переоценка = `0`;
- все изменение резерва относится в движение портфеля.

Все денежные значения переводятся в **млн BYN**.


In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)


## 1. Настройки

In [ ]:
# ------------------------------------------------------------
# ФАЙЛЫ
# ------------------------------------------------------------

PORTFOLIO_1_FILE = Path("portfolio_1.xlsx")
PORTFOLIO_2_FILE = Path("portfolio_2.xlsx")

SHEET_1 = 0
SHEET_2 = 0

OUTPUT_FILE = Path("factor_analysis.xlsx")


# ------------------------------------------------------------
# ЕДИНИЦЫ ИЗМЕРЕНИЯ
# ------------------------------------------------------------

# По условию задолженность и резерв в исходнике делим на 1000,
# чтобы получить млн BYN.
MONEY_DIVISOR = 1000

# Если в Excel % резерва записан как 0.005 для 0.5%,
# поставьте True.
#
# Если в колонке записано 0.5, 5, 20, 30 и т.д.,
# оставьте False.
RESERVE_RATE_STORED_AS_FRACTION = False


# ------------------------------------------------------------
# ВАЛЮТА
# ------------------------------------------------------------

# Для BYN курс принимаем равным 1.
BYN_CODES = {
    "BYN",
    "933",
    "БЕЛОРУССКИЙРУБЛЬ",
    "БЕЛРУБ",
    "БЕЛОРУБ",
}


# ------------------------------------------------------------
# ПРОВЕРКИ
# ------------------------------------------------------------

# При True код остановится, если для существующего договора
# отсутствуют обязательные числовые данные.
STRICT_VALIDATION = True


## 2. Возможные названия колонок

Код пытается сам найти колонки. Если в ваших файлах название отличается, достаточно добавить его в соответствующий список ниже.


In [ ]:
COLUMN_ALIASES = {
    "unp": [
        "УНП",
        "УНП клиента",
        "УНП заемщика",
        "УНП заёмщика",
    ],

    "contract": [
        "Номер договора",
        "№ договора",
        "Номер кредитного договора",
        "Договор",
    ],

    "operation": [
        "Тип операции",
        "Актив/УО",
        "Актив (УО)",
        "Актив или УО",
        "Актив/УО (тип операции)",
    ],

    "currency": [
        "Валюта",
        "Код валюты",
        "Валюта договора",
    ],

    "debt": [
        "Задолженность тыс. BYN",
        "Задолженность, тыс. BYN",
        "Задолженность тыс BYN",
        "Задолженность",
    ],

    "reserve": [
        "Резерв тыс. BYN",
        "Резерв, тыс. BYN",
        "Резерв тыс BYN",
        "Резерв",
    ],

    "fx": [
        "Курс валюты",
        "Курс",
        "Курс валют",
    ],

    "gr": [
        "ГР",
        "Группа риска",
    ],

    "reserve_rate": [
        "%рез",
        "% рез",
        "%резерва",
        "% резерва",
        "Процент резерва",
    ],

    "ni": [
        "НИ",
    ],

    "pfn": [
        "ПФН",
    ],

    "restra": [
        "Рестра",
        "рестра",
        "Реструктуризация",
        "Реструкт",
    ],

    "collateral": [
        "Обеспеченность",
    ],
}


## 3. Вспомогательные функции

In [ ]:
def normalize_header(value):
    """Нормализация заголовка колонки."""
    s = str(value).replace("\xa0", " ").replace("\n", " ").replace("\r", " ")
    s = s.strip().lower().replace("ё", "е")
    s = re.sub(r"\s+", " ", s)
    return s


def resolve_columns(df, aliases):
    """
    Находит обязательные колонки по списку допустимых названий.
    Поиск намеренно делается по точному нормализованному совпадению,
    чтобы, например, 'Резерв' не перепутался с '% резерва'.
    """
    actual = {normalize_header(c): c for c in df.columns}

    result = {}
    missing = {}

    for field, variants in aliases.items():
        found = None

        for variant in variants:
            key = normalize_header(variant)
            if key in actual:
                found = actual[key]
                break

        if found is None:
            missing[field] = variants
        else:
            result[field] = found

    if missing:
        text = ["Не найдены обязательные колонки:"]
        for field, variants in missing.items():
            text.append(f"  - {field}: {variants}")

        text.append("\nКолонки в файле:")
        text.extend([f"  - {c}" for c in df.columns])

        raise KeyError("\n".join(text))

    return result


def clean_id(value):
    """
    Приводит УНП/номер договора к строке.
    Убирает типичный хвост '.0', появляющийся после чтения Excel.
    """
    if pd.isna(value):
        return ""

    if isinstance(value, (int, np.integer)):
        return str(value)

    if isinstance(value, (float, np.floating)) and float(value).is_integer():
        return str(int(value))

    s = str(value).strip().replace("\xa0", "")
    if re.fullmatch(r"\d+\.0", s):
        s = s[:-2]

    return s


def normalize_operation(value):
    """Нормализует тип операции до 'Актив' или 'УО'."""
    if pd.isna(value):
        return ""

    s = str(value).strip().lower().replace("ё", "е")
    s_no_space = re.sub(r"\s+", "", s)

    if "уо" in s_no_space:
        return "УО"

    if "актив" in s:
        return "Актив"

    return str(value).strip()


def normalize_currency(value):
    if pd.isna(value):
        return ""

    s = clean_id(value).upper()
    s = re.sub(r"\s+", "", s)
    return s


def parse_number(series):
    """Преобразование числового поля с пробелами и запятыми."""
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce")

    s = (
        series.astype(str)
        .str.replace("\xa0", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False)
        .str.replace("%", "", regex=False)
    )

    s = s.replace({
        "": np.nan,
        "nan": np.nan,
        "None": np.nan,
        "none": np.nan,
    })

    return pd.to_numeric(s, errors="coerce")


def parse_flag(series):
    """
    НИ / ПФН / рестра -> 0/1.
    Пустое значение остается NaN, чтобы отсутствие данных
    не маскировалось под нулевой признак.
    """
    s = (
        series.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(",", ".", regex=False)
    )

    mapping = {
        "1": 1.0,
        "1.0": 1.0,
        "да": 1.0,
        "yes": 1.0,
        "true": 1.0,

        "0": 0.0,
        "0.0": 0.0,
        "нет": 0.0,
        "no": 0.0,
        "false": 0.0,

        "": np.nan,
        "nan": np.nan,
        "none": np.nan,
    }

    out = s.map(mapping)

    numeric = pd.to_numeric(s, errors="coerce")
    out = out.fillna(numeric)

    return out


def normalize_collateral(value):
    """
    Для факторного анализа сохраняем три технических состояния:
      1) Обеспеченный
      2) Недостаточно обеспеченный
      3) Необеспеченный

    Для хорошего/среднего/плохого портфеля позже используется
    только разделение 'Необеспеченный' / 'Остальные'.
    """
    if pd.isna(value):
        return np.nan

    s = str(value).strip().lower().replace("ё", "е")
    s = re.sub(r"\s+", " ", s)

    if not s:
        return np.nan

    # Сначала обязательно проверяем "недостаточно".
    if "недостаточно" in s and "обеспеч" in s:
        return "Недостаточно обеспеченный"

    if re.search(r"\bне\s*обеспеч", s) or "необеспеч" in s:
        return "Необеспеченный"

    return "Обеспеченный"


def is_byn_currency(currency):
    if pd.isna(currency):
        return False

    s = normalize_currency(currency)
    return s in BYN_CODES


def make_key(df):
    return (
        df["УНП"].astype(str)
        + "/"
        + df["Номер договора"].astype(str)
        + "/"
        + df["Тип операции"].astype(str)
        + "/"
        + df["Валюта"].astype(str)
    )


def logical_risk_group(operation, ni, pfn, collateral):
    """
    Промежуточная ГР по согласованному алгоритму.

    Без НИ и ПФН:
      обеспеченный -> 1
      недостаточно обеспеченный / необеспеченный -> 2

    НИ без ПФН:
      обеспеченный / недостаточно обеспеченный -> 2
      необеспеченный -> 3

    ПФН:
      обеспеченный / недостаточно обеспеченный -> 3
      необеспеченный:
          Актив -> 4
          УО    -> 3
    """
    if pd.isna(ni) or pd.isna(pfn) or pd.isna(collateral):
        return np.nan

    ni = int(ni)
    pfn = int(pfn)

    unsecured = collateral == "Необеспеченный"
    insufficient = collateral == "Недостаточно обеспеченный"

    if pfn == 1:
        if unsecured:
            return 4 if operation == "Актив" else 3
        return 3

    if ni == 1:
        if unsecured:
            return 3
        return 2

    # нет НИ и нет ПФН
    if insufficient or unsecured:
        return 2

    return 1


STANDARD_RATE_BY_GR = {
    1: 0.5,
    2: 5.0,
    3: 20.0,
    4: 30.0,
    5: 50.0,
    6: 100.0,
}


def standard_rate(gr):
    if pd.isna(gr):
        return np.nan

    try:
        return STANDARD_RATE_BY_GR.get(int(gr), np.nan)
    except Exception:
        return np.nan


def same_with_nan(a, b):
    """NaN == NaN для целей сравнения состояния."""
    if pd.isna(a) and pd.isna(b):
        return True
    return a == b


## 4. Подготовка одного портфеля

В этой функции:

- создается единый ключ;
- задолженность и резерв переводятся в млн BYN;
- `%рез` приводится к процентным пунктам;
- для BYN курс выставляется `1`;
- проверяется отсутствие дублей ключа.


In [ ]:
def prepare_portfolio(df_raw, portfolio_number):
    cols = resolve_columns(df_raw, COLUMN_ALIASES)

    print(f"\nПортфель {portfolio_number}: найденные колонки")
    display(
        pd.DataFrame({
            "Поле": list(cols.keys()),
            "Колонка": list(cols.values()),
        })
    )

    out = pd.DataFrame(index=df_raw.index)

    out["УНП"] = df_raw[cols["unp"]].map(clean_id)
    out["Номер договора"] = df_raw[cols["contract"]].map(clean_id)
    out["Тип операции"] = df_raw[cols["operation"]].map(normalize_operation)
    out["Валюта"] = df_raw[cols["currency"]].map(normalize_currency)

    out["Задолженность"] = parse_number(df_raw[cols["debt"]]) / MONEY_DIVISOR
    out["Резерв"] = parse_number(df_raw[cols["reserve"]]) / MONEY_DIVISOR
    out["Курс"] = parse_number(df_raw[cols["fx"]])
    out["ГР"] = parse_number(df_raw[cols["gr"]])
    out["%рез"] = parse_number(df_raw[cols["reserve_rate"]])

    if RESERVE_RATE_STORED_AS_FRACTION:
        out["%рез"] = out["%рез"] * 100

    out["НИ"] = parse_flag(df_raw[cols["ni"]])
    out["ПФН"] = parse_flag(df_raw[cols["pfn"]])
    out["рестра"] = parse_flag(df_raw[cols["restra"]])

    # Сохраняем исходное значение и техническую категорию.
    out["Обеспеченность"] = df_raw[cols["collateral"]]
    out["_Обеспеченность_кат"] = out["Обеспеченность"].map(normalize_collateral)

    # Курс BYN = 1 только для реально существующей строки.
    byn_mask = out["Валюта"].map(is_byn_currency)
    out.loc[byn_mask, "Курс"] = 1.0

    out["Ключ"] = make_key(out)

    # Проверка обязательных частей ключа.
    bad_key = (
        out["УНП"].eq("")
        | out["Номер договора"].eq("")
        | out["Тип операции"].eq("")
        | out["Валюта"].eq("")
    )

    if bad_key.any():
        sample = out.loc[
            bad_key,
            ["УНП", "Номер договора", "Тип операции", "Валюта"]
        ].head(20)

        raise ValueError(
            f"В портфеле {portfolio_number} есть строки "
            f"с неполным ключом.\nПример:\n{sample}"
        )

    # Проверка типа операции.
    invalid_operation = ~out["Тип операции"].isin(["Актив", "УО"])

    if invalid_operation.any():
        values = out.loc[
            invalid_operation,
            "Тип операции"
        ].drop_duplicates().tolist()

        raise ValueError(
            f"В портфеле {portfolio_number} найдены неизвестные "
            f"типы операции: {values}"
        )

    # По условию дубли ключа не ожидаются.
    duplicates = out["Ключ"].duplicated(keep=False)

    if duplicates.any():
        sample = out.loc[
            duplicates,
            [
                "Ключ",
                "УНП",
                "Номер договора",
                "Тип операции",
                "Валюта",
            ]
        ].sort_values("Ключ").head(30)

        raise ValueError(
            f"В портфеле {portfolio_number} обнаружены дубли ключа. "
            f"По условию ключ должен быть уникальным.\n\n{sample}"
        )

    return out


## 5. Читаем и подготавливаем оба портфеля

In [ ]:
portfolio_1_raw = pd.read_excel(
    PORTFOLIO_1_FILE,
    sheet_name=SHEET_1,
)

portfolio_2_raw = pd.read_excel(
    PORTFOLIO_2_FILE,
    sheet_name=SHEET_2,
)

p1 = prepare_portfolio(portfolio_1_raw, 1)
p2 = prepare_portfolio(portfolio_2_raw, 2)

print(f"Портфель 1: {len(p1):,} уникальных ключей")
print(f"Портфель 2: {len(p2):,} уникальных ключей")


## 6. Outer join двух портфелей

В результате остаются **все уникальные ключи из обоих файлов**.

Для отсутствующей стороны:

- задолженность = `0`;
- резерв = `0`;
- ГР = `0`;
- `%рез` = `0`;
- курс = пусто;
- НИ / ПФН / рестра / обеспеченность = пусто.


In [ ]:
key_cols = [
    "Ключ",
    "УНП",
    "Номер договора",
    "Тип операции",
    "Валюта",
]

value_cols = [
    "Задолженность",
    "Резерв",
    "Курс",
    "ГР",
    "%рез",
    "НИ",
    "ПФН",
    "рестра",
    "Обеспеченность",
    "_Обеспеченность_кат",
]

p1_merge = p1[key_cols + value_cols].copy()
p2_merge = p2[key_cols + value_cols].copy()

p1_merge["_Есть_1"] = True
p2_merge["_Есть_2"] = True

# Ключевые поля берем один раз.
merged = p1_merge.merge(
    p2_merge,
    on=key_cols,
    how="outer",
    suffixes=("_1", "_2"),
)

merged["_Есть_1"] = merged["_Есть_1"].eq(True)
merged["_Есть_2"] = merged["_Есть_2"].eq(True)


# ------------------------------------------------------------
# Подстановка значений ТОЛЬКО для отсутствующего ключа.
# Если строка существует, но само поле пустое, мы это не маскируем.
# ------------------------------------------------------------

zero_if_missing_key = [
    "Задолженность",
    "Резерв",
    "ГР",
    "%рез",
]

for field in zero_if_missing_key:
    merged.loc[~merged["_Есть_1"], f"{field}_1"] = 0.0
    merged.loc[~merged["_Есть_2"], f"{field}_2"] = 0.0


# Курс/флаги/обеспеченность при отсутствии ключа остаются пустыми.
# Для существующих BYN-договоров курс гарантированно равен 1
# еще на этапе prepare_portfolio().

print(f"Всего уникальных ключей после объединения: {len(merged):,}")

print(
    "Есть в обоих:",
    int((merged["_Есть_1"] & merged["_Есть_2"]).sum())
)

print(
    "Только в первом:",
    int((merged["_Есть_1"] & ~merged["_Есть_2"]).sum())
)

print(
    "Только во втором:",
    int((~merged["_Есть_1"] & merged["_Есть_2"]).sum())
)


## 7. Проверка данных перед факторным анализом

In [ ]:
def validate_merged_data(df):
    problems = []

    # Для реально существующих строк эти поля нужны для основных формул.
    required_numeric = [
        "Задолженность",
        "Резерв",
        "ГР",
        "%рез",
    ]

    for side in [1, 2]:
        exists = df[f"_Есть_{side}"]

        for field in required_numeric:
            col = f"{field}_{side}"
            bad = exists & df[col].isna()

            if bad.any():
                problems.append(
                    f"{col}: {int(bad.sum())} пустых значений "
                    f"у существующих ключей"
                )

    # Курс требуется у валютной строки, если строка реально существует.
    for side in [1, 2]:
        exists = df[f"_Есть_{side}"]
        non_byn = ~df["Валюта"].map(is_byn_currency)

        bad_fx = exists & non_byn & df[f"Курс_{side}"].isna()

        if bad_fx.any():
            problems.append(
                f"Курс_{side}: {int(bad_fx.sum())} пустых курсов "
                f"для существующих не-BYN ключей"
            )

    if problems:
        message = "Обнаружены проблемы с обязательными данными:\n" + "\n".join(
            f"  - {x}" for x in problems
        )

        if STRICT_VALIDATION:
            raise ValueError(message)
        else:
            warnings.warn(message)

    return problems


validation_problems = validate_merged_data(merged)

if not validation_problems:
    print("Проверка обязательных числовых данных пройдена.")


## 8. Вычисляем задолженность в валюте

Так как в исходнике ее нет:

`Задолженность в валюте = Задолженность в BYN / Курс`

Задолженность уже находится в **млн BYN**, поэтому валютная задолженность получается в млн единиц валюты.


In [ ]:
for side in [1, 2]:
    debt_col = f"Задолженность_{side}"
    fx_col = f"Курс_{side}"

    merged[f"Задолженность_в_валюте_{side}"] = np.where(
        merged[f"_Есть_{side}"]
        & merged[fx_col].notna()
        & merged[fx_col].ne(0),
        merged[debt_col] / merged[fx_col],
        np.nan,
    )


## 9. Основные факторы

In [ ]:
# ------------------------------------------------------------
# 1. Расходы на резервы
# ------------------------------------------------------------

merged["Расходы на резервы"] = (
    merged["Резерв_1"]
    - merged["Резерв_2"]
)


# ------------------------------------------------------------
# 2. Изменение качества
#
# Если ключ новый (нет в портфеле 1) -> 0.
# Если одного из периодов нет -> 0.
# Если ГР не изменилась -> 0.
# ------------------------------------------------------------

both_exist = merged["_Есть_1"] & merged["_Есть_2"]

merged["Изменение качества"] = np.where(
    both_exist
    & merged["ГР_1"].ne(merged["ГР_2"]),
    -merged["Задолженность_2"]
    * (
        merged["%рез_2"]
        - merged["%рез_1"]
    )
    / 100,
    0.0,
)


# ------------------------------------------------------------
# 3. Переоценка
#
# Считается только если ключ присутствует в обоих портфелях.
# Для BYN курс = 1, поэтому фактор автоматически равен 0.
# ------------------------------------------------------------

merged["Переоценка"] = np.where(
    both_exist,
    -merged["Задолженность_в_валюте_1"]
    * (
        merged["Курс_2"]
        - merged["Курс_1"]
    )
    * merged["%рез_1"]
    / 100,
    0.0,
)

# Для случаев, где фактор по определению не считается,
# NaN превращаем в 0.
merged.loc[~both_exist, "Переоценка"] = 0.0


# ------------------------------------------------------------
# 4. Движение портфеля
# ------------------------------------------------------------

merged["Движение портфеля"] = (
    -(
        merged["Задолженность_2"]
        * merged["%рез_1"]
        / 100
        -
        merged["Задолженность_1"]
        * merged["%рез_1"]
        / 100
    )
    - merged["Переоценка"]
)


# ------------------------------------------------------------
# НОВЫЙ ДОГОВОР:
# если ключа не было в 1-м портфеле, все расходы на резерв
# относятся в движение портфеля.
# ------------------------------------------------------------

new_key = (
    ~merged["_Есть_1"]
    & merged["_Есть_2"]
)

merged.loc[new_key, "Изменение качества"] = 0.0
merged.loc[new_key, "Переоценка"] = 0.0
merged.loc[new_key, "Движение портфеля"] = (
    merged.loc[new_key, "Расходы на резервы"]
)


# ------------------------------------------------------------
# ВЫБЫВШИЙ ДОГОВОР:
# во втором портфеле курса уже нет, переоценку не считаем.
# ------------------------------------------------------------

exited_key = (
    merged["_Есть_1"]
    & ~merged["_Есть_2"]
)

merged.loc[exited_key, "Изменение качества"] = 0.0
merged.loc[exited_key, "Переоценка"] = 0.0

# Формула движения для выбывшего ключа остается расчетной:
# + Задолженность_1 * %рез_1 / 100.


## 10. Разложение движения портфеля

Для классификации используется:

- состояние второго портфеля, если договор в нем есть;
- состояние первого портфеля, если договор полностью выбыл.

Приоритет классификации: **Плохой → Средний → Хороший**.

### Плохой портфель

Договор относится к плохому портфелю, если выполняется **хотя бы одно** условие:

- `ПФН = 1` и обеспеченность = `Необеспеченный`;
- `ГР = 5` или `ГР = 6`;
- `Рестра = 1`.

### Средний портфель

Если договор не попал в плохой портфель, он относится к среднему, если:

- `ПФН = 1` и обеспеченность любая, кроме `Необеспеченный`;
- **или**
- `НИ = 1` и обеспеченность = `Необеспеченный`.

### Хороший портфель

Все остальные случаи.

`Недостаточно обеспеченный` для этой классификации относится к категории
**«любая обеспеченность кроме необеспеченного»**.


In [ ]:
# Выбираем состояние для классификации движения.
# Если договор есть во втором портфеле -> используем состояние второго.
# Если договор полностью выбыл -> используем состояние первого.
use_second = merged["_Есть_2"]

merged["_НИ_для_движения"] = np.where(
    use_second,
    merged["НИ_2"],
    merged["НИ_1"],
)

merged["_ПФН_для_движения"] = np.where(
    use_second,
    merged["ПФН_2"],
    merged["ПФН_1"],
)

merged["_Рестра_для_движения"] = np.where(
    use_second,
    merged["рестра_2"],
    merged["рестра_1"],
)

merged["_ГР_для_движения"] = np.where(
    use_second,
    merged["ГР_2"],
    merged["ГР_1"],
)

merged["_Обеспеченность_для_движения"] = np.where(
    use_second,
    merged["_Обеспеченность_кат_2"],
    merged["_Обеспеченность_кат_1"],
)


# ------------------------------------------------------------
# Обеспеченность
# ------------------------------------------------------------

is_unsecured = (
    merged["_Обеспеченность_для_движения"]
    == "Необеспеченный"
)

is_other_collateral = (
    merged["_Обеспеченность_для_движения"].notna()
    & ~is_unsecured
)


# ------------------------------------------------------------
# ПЛОХОЙ ПОРТФЕЛЬ
#
# 1. ПФН + необеспеченный
# ИЛИ
# 2. ГР 5-6
# ИЛИ
# 3. рестра
# ------------------------------------------------------------

is_bad = (
    (
        merged["_ПФН_для_движения"].eq(1)
        & is_unsecured
    )
    |
    merged["_ГР_для_движения"].isin([5, 6])
    |
    merged["_Рестра_для_движения"].eq(1)
)


# ------------------------------------------------------------
# СРЕДНИЙ ПОРТФЕЛЬ
#
# 1. ПФН + любая обеспеченность, кроме необеспеченного
# ИЛИ
# 2. НИ + необеспеченный
#
# Плохой имеет более высокий приоритет.
# ------------------------------------------------------------

is_medium_raw = (
    (
        merged["_ПФН_для_движения"].eq(1)
        & is_other_collateral
    )
    |
    (
        merged["_НИ_для_движения"].eq(1)
        & is_unsecured
    )
)

is_medium = (
    ~is_bad
    & is_medium_raw
)


# ------------------------------------------------------------
# ХОРОШИЙ ПОРТФЕЛЬ
# ------------------------------------------------------------

merged["Категория движения"] = np.select(
    [
        is_bad,
        is_medium,
    ],
    [
        "Плохой",
        "Средний",
    ],
    default="Хороший",
)


# ------------------------------------------------------------
# Разносим фактор движения по трем категориям
# ------------------------------------------------------------

merged["Движение хорошего портфеля"] = np.where(
    merged["Категория движения"].eq("Хороший"),
    merged["Движение портфеля"],
    0.0,
)

merged["Движение среднего портфеля"] = np.where(
    merged["Категория движения"].eq("Средний"),
    merged["Движение портфеля"],
    0.0,
)

merged["Движение плохого портфеля"] = np.where(
    merged["Категория движения"].eq("Плохой"),
    merged["Движение портфеля"],
    0.0,
)


# ------------------------------------------------------------
# Контроль количества договоров по категориям
# ------------------------------------------------------------

display(
    merged["Категория движения"]
    .value_counts(dropna=False)
    .rename_axis("Категория")
    .reset_index(name="Количество ключей")
)


## 11. Разложение изменения качества

Выделяем:

1. **Изменение финсостояния** — поменялся НИ и/или ПФН.
2. **Изменение обеспеченности**.
3. **Прочее изменение качества**.

Если изменился только один из факторов — все изменение качества относится в него.

Если одновременно изменились и финсостояние, и обеспеченность, используется промежуточный путь:

**старое состояние → новая обеспеченность → новое финсостояние**

Например:

`ГР1 (обеспеченный, без НИ/ПФН)`  
→ смена обеспеченности → `ГР2`  
→ появление ПФН → `ГР4`.

Для такого одновременного перехода расчет производится по нормативным ставкам ГР:

| ГР | % резерва |
|---:|---:|
| 1 | 0.5% |
| 2 | 5% |
| 3 | 20% |
| 4 | 30% |
| 5 | 50% |
| 6 | 100% |

Если фактические `%рез` отличаются от нормативных ставок, разница автоматически попадает в **«Прочее изменение качества»**, поэтому внутреннее разложение всегда сходится с общим фактором качества.


In [ ]:
# Служебные логические ГР.
merged["_Логическая_ГР_1"] = merged.apply(
    lambda r: logical_risk_group(
        r["Тип операции"],
        r["НИ_1"],
        r["ПФН_1"],
        r["_Обеспеченность_кат_1"],
    ),
    axis=1,
)

merged["_Логическая_ГР_2"] = merged.apply(
    lambda r: logical_risk_group(
        r["Тип операции"],
        r["НИ_2"],
        r["ПФН_2"],
        r["_Обеспеченность_кат_2"],
    ),
    axis=1,
)


# ------------------------------------------------------------
# Изменился ли финансовый статус?
# Сравнивается именно пара (НИ, ПФН).
# ------------------------------------------------------------

merged["_Изменилось_финсостояние"] = merged.apply(
    lambda r: (
        not same_with_nan(r["НИ_1"], r["НИ_2"])
        or not same_with_nan(r["ПФН_1"], r["ПФН_2"])
    ),
    axis=1,
)


# ------------------------------------------------------------
# Изменилась ли обеспеченность?
# Используем 3 технических состояния.
# ------------------------------------------------------------

merged["_Изменилась_обеспеченность"] = merged.apply(
    lambda r: not same_with_nan(
        r["_Обеспеченность_кат_1"],
        r["_Обеспеченность_кат_2"],
    ),
    axis=1,
)


# Инициализация.
merged["Изменение финсостояния"] = 0.0
merged["Изменение обеспеченности"] = 0.0
merged["Прочее изменение качества"] = 0.0


# Качество раскладываем только по договорам,
# которые присутствуют в обоих портфелях.
quality_rows = both_exist & merged["Изменение качества"].ne(0)


# ------------------------------------------------------------
# Сценарий 1: изменилось только финсостояние.
# ------------------------------------------------------------

mask = (
    quality_rows
    & merged["_Изменилось_финсостояние"]
    & ~merged["_Изменилась_обеспеченность"]
)

merged.loc[
    mask,
    "Изменение финсостояния"
] = merged.loc[
    mask,
    "Изменение качества"
]


# ------------------------------------------------------------
# Сценарий 2: изменилась только обеспеченность.
# ------------------------------------------------------------

mask = (
    quality_rows
    & ~merged["_Изменилось_финсостояние"]
    & merged["_Изменилась_обеспеченность"]
)

merged.loc[
    mask,
    "Изменение обеспеченности"
] = merged.loc[
    mask,
    "Изменение качества"
]


# ------------------------------------------------------------
# Сценарий 3: ни НИ/ПФН, ни обеспеченность не изменились,
# но фактическая ГР изменилась.
# ------------------------------------------------------------

mask = (
    quality_rows
    & ~merged["_Изменилось_финсостояние"]
    & ~merged["_Изменилась_обеспеченность"]
)

merged.loc[
    mask,
    "Прочее изменение качества"
] = merged.loc[
    mask,
    "Изменение качества"
]


# ------------------------------------------------------------
# Сценарий 4:
# одновременно изменилось финсостояние и обеспеченность.
# ------------------------------------------------------------

both_changed = (
    quality_rows
    & merged["_Изменилось_финсостояние"]
    & merged["_Изменилась_обеспеченность"]
)


def split_both_quality_changes(row):
    """
    Шаг 1:
        старое финсостояние + старая обеспеченность
        ->
        старое финсостояние + новая обеспеченность

    Шаг 2:
        старое финсостояние + новая обеспеченность
        ->
        новое финсостояние + новая обеспеченность
    """
    old_gr = logical_risk_group(
        row["Тип операции"],
        row["НИ_1"],
        row["ПФН_1"],
        row["_Обеспеченность_кат_1"],
    )

    middle_gr = logical_risk_group(
        row["Тип операции"],
        row["НИ_1"],
        row["ПФН_1"],
        row["_Обеспеченность_кат_2"],
    )

    final_gr = logical_risk_group(
        row["Тип операции"],
        row["НИ_2"],
        row["ПФН_2"],
        row["_Обеспеченность_кат_2"],
    )

    old_rate = standard_rate(old_gr)
    middle_rate = standard_rate(middle_gr)
    final_rate = standard_rate(final_gr)

    if (
        pd.isna(old_rate)
        or pd.isna(middle_rate)
        or pd.isna(final_rate)
    ):
        # Если невозможно построить промежуточную ГР,
        # весь фактор относим в "прочее", не выдумывая раскладку.
        return pd.Series({
            "_Промежуточная_ГР": middle_gr,
            "_Качество_обеспеченность_расчет": 0.0,
            "_Качество_финсостояние_расчет": 0.0,
            "_Качество_прочее_расчет": row["Изменение качества"],
        })

    debt_2 = row["Задолженность_2"]

    collateral_component = (
        -debt_2
        * (middle_rate - old_rate)
        / 100
    )

    financial_component = (
        -debt_2
        * (final_rate - middle_rate)
        / 100
    )

    other_component = (
        row["Изменение качества"]
        - collateral_component
        - financial_component
    )

    return pd.Series({
        "_Промежуточная_ГР": middle_gr,
        "_Качество_обеспеченность_расчет": collateral_component,
        "_Качество_финсостояние_расчет": financial_component,
        "_Качество_прочее_расчет": other_component,
    })


if both_changed.any():
    split_result = merged.loc[both_changed].apply(
        split_both_quality_changes,
        axis=1,
    )

    merged.loc[
        both_changed,
        "_Промежуточная_ГР"
    ] = split_result["_Промежуточная_ГР"]

    merged.loc[
        both_changed,
        "Изменение обеспеченности"
    ] = split_result["_Качество_обеспеченность_расчет"]

    merged.loc[
        both_changed,
        "Изменение финсостояния"
    ] = split_result["_Качество_финсостояние_расчет"]

    merged.loc[
        both_changed,
        "Прочее изменение качества"
    ] = split_result["_Качество_прочее_расчет"]


# Новые/выбывшие ключи не имеют фактора качества.
merged.loc[
    ~both_exist,
    [
        "Изменение финсостояния",
        "Изменение обеспеченности",
        "Прочее изменение качества",
    ]
] = 0.0


## 12. Контроль сходимости

In [ ]:
# ------------------------------------------------------------
# Основные факторы
# ------------------------------------------------------------

merged["Контроль основных факторов"] = (
    merged["Расходы на резервы"]
    - (
        merged["Изменение качества"]
        + merged["Переоценка"]
        + merged["Движение портфеля"]
    )
)


# ------------------------------------------------------------
# Разложение движения
# ------------------------------------------------------------

merged["Контроль движения"] = (
    merged["Движение портфеля"]
    - (
        merged["Движение хорошего портфеля"]
        + merged["Движение среднего портфеля"]
        + merged["Движение плохого портфеля"]
    )
)


# ------------------------------------------------------------
# Разложение качества
# ------------------------------------------------------------

merged["Контроль качества"] = (
    merged["Изменение качества"]
    - (
        merged["Изменение финсостояния"]
        + merged["Изменение обеспеченности"]
        + merged["Прочее изменение качества"]
    )
)


control = pd.DataFrame({
    "Показатель": [
        "Количество ключей",
        "Есть в обоих портфелях",
        "Только в портфеле 1",
        "Только в портфеле 2",
        "Расходы на резервы",
        "Изменение качества",
        "Переоценка",
        "Движение портфеля",
        "Контроль основных факторов",
        "Контроль движения",
        "Контроль качества",
    ],
    "Значение": [
        len(merged),
        int((merged["_Есть_1"] & merged["_Есть_2"]).sum()),
        int((merged["_Есть_1"] & ~merged["_Есть_2"]).sum()),
        int((~merged["_Есть_1"] & merged["_Есть_2"]).sum()),
        merged["Расходы на резервы"].sum(),
        merged["Изменение качества"].sum(),
        merged["Переоценка"].sum(),
        merged["Движение портфеля"].sum(),
        merged["Контроль основных факторов"].sum(),
        merged["Контроль движения"].sum(),
        merged["Контроль качества"].sum(),
    ],
})

display(control)


### Важное замечание о контроле основных факторов

`Контроль основных факторов` может быть ненулевым, если фактический резерв в портфеле не равен точно:

`Задолженность × %рез / 100`

или если бизнес-правила расчета резерва содержат дополнительные компоненты, которых нет в текущей факторной модели.

Код **не прячет** такую разницу и показывает ее отдельным контрольным столбцом.


## 13. Формируем итоговую детализацию

In [ ]:
output_columns = [
    # Ключ
    "Ключ",
    "УНП",
    "Номер договора",
    "Тип операции",
    "Валюта",

    # Признаки присутствия
    "_Есть_1",
    "_Есть_2",

    # Задолженность
    "Задолженность_1",
    "Задолженность_2",

    # Валютная задолженность для аудита переоценки
    "Задолженность_в_валюте_1",
    "Задолженность_в_валюте_2",

    # Резерв
    "Резерв_1",
    "Резерв_2",

    # Курс
    "Курс_1",
    "Курс_2",

    # ГР
    "ГР_1",
    "ГР_2",

    # % рез
    "%рез_1",
    "%рез_2",

    # НИ
    "НИ_1",
    "НИ_2",

    # ПФН
    "ПФН_1",
    "ПФН_2",

    # Рестра
    "рестра_1",
    "рестра_2",

    # Обеспеченность
    "Обеспеченность_1",
    "Обеспеченность_2",

    # Техническая категория обеспеченности
    "_Обеспеченность_кат_1",
    "_Обеспеченность_кат_2",

    # Основные факторы
    "Расходы на резервы",
    "Изменение качества",
    "Переоценка",
    "Движение портфеля",

    # Декомпозиция движения
    "Категория движения",
    "Движение хорошего портфеля",
    "Движение среднего портфеля",
    "Движение плохого портфеля",

    # Декомпозиция качества
    "Изменение финсостояния",
    "Изменение обеспеченности",
    "Прочее изменение качества",

    # Для аудита сложных переходов
    "_Логическая_ГР_1",
    "_Промежуточная_ГР",
    "_Логическая_ГР_2",
    "_Изменилось_финсостояние",
    "_Изменилась_обеспеченность",

    # Контроли
    "Контроль основных факторов",
    "Контроль движения",
    "Контроль качества",
]

detail = (
    merged[output_columns]
    .sort_values(
        [
            "Тип операции",
            "УНП",
            "Номер договора",
            "Валюта",
        ]
    )
    .reset_index(drop=True)
)

display(detail.head(20))


## 14. Свод факторов

In [ ]:
factor_columns = [
    "Расходы на резервы",
    "Изменение качества",
    "Переоценка",
    "Движение портфеля",
    "Движение хорошего портфеля",
    "Движение среднего портфеля",
    "Движение плохого портфеля",
    "Изменение финсостояния",
    "Изменение обеспеченности",
    "Прочее изменение качества",
    "Контроль основных факторов",
    "Контроль движения",
    "Контроль качества",
]

summary_by_operation = (
    detail
    .groupby(
        "Тип операции",
        dropna=False,
    )[factor_columns]
    .sum()
    .reset_index()
)

total_row = {
    "Тип операции": "ИТОГО",
}

for col in factor_columns:
    total_row[col] = detail[col].sum()

summary_by_operation = pd.concat(
    [
        summary_by_operation,
        pd.DataFrame([total_row]),
    ],
    ignore_index=True,
)

display(summary_by_operation)


## 15. Самопроверка логики сложного перехода

Проверяем пример:

- Актив;
- было: обеспеченный, без НИ и ПФН → логическая ГР1;
- стало: необеспеченный, ПФН=1 → логическая ГР4.

Ожидаемый промежуточный путь:

`ГР1 → ГР2` за счет обеспеченности, затем `ГР2 → ГР4` за счет финсостояния.


In [ ]:
test_row = pd.Series({
    "Тип операции": "Актив",

    "НИ_1": 0,
    "ПФН_1": 0,
    "_Обеспеченность_кат_1": "Обеспеченный",

    "НИ_2": 0,
    "ПФН_2": 1,
    "_Обеспеченность_кат_2": "Необеспеченный",

    "Задолженность_2": 100.0,

    # При ставках 0.5% -> 30%:
    "Изменение качества": -100.0 * (30.0 - 0.5) / 100,
})

test_split = split_both_quality_changes(test_row)

assert logical_risk_group(
    "Актив", 0, 0, "Обеспеченный"
) == 1

assert test_split["_Промежуточная_ГР"] == 2

assert logical_risk_group(
    "Актив", 0, 1, "Необеспеченный"
) == 4

# ГР1 -> ГР2: 0.5% -> 5%
assert np.isclose(
    test_split["_Качество_обеспеченность_расчет"],
    -4.5,
)

# ГР2 -> ГР4: 5% -> 30%
assert np.isclose(
    test_split["_Качество_финсостояние_расчет"],
    -25.0,
)

assert np.isclose(
    test_split["_Качество_прочее_расчет"],
    0.0,
)

print("Самопроверка сложного перехода успешно пройдена.")
display(test_split.to_frame("Значение"))


## 16. Сохраняем итоговый Excel

Файл содержит:

- **Детализация** — все уникальные ключи и все исходные/расчетные показатели;
- **Свод факторов** — суммы по Активу и УО;
- **Контроль** — общие контрольные суммы;
- **Ставки ГР** — справочник нормативных ставок.

Сохранение выполняется через `openpyxl`, без зависимости от `xlsxwriter`.


In [ ]:
rates_reference = pd.DataFrame({
    "ГР": list(STANDARD_RATE_BY_GR.keys()),
    "Нормативный % резерва": list(STANDARD_RATE_BY_GR.values()),
})

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl",
) as writer:

    detail.to_excel(
        writer,
        sheet_name="Детализация",
        index=False,
    )

    summary_by_operation.to_excel(
        writer,
        sheet_name="Свод факторов",
        index=False,
    )

    control.to_excel(
        writer,
        sheet_name="Контроль",
        index=False,
    )

    rates_reference.to_excel(
        writer,
        sheet_name="Ставки ГР",
        index=False,
    )

    # Простое форматирование через openpyxl
    for sheet_name in [
        "Детализация",
        "Свод факторов",
        "Контроль",
        "Ставки ГР",
    ]:
        ws = writer.book[sheet_name]
        ws.freeze_panes = "A2"

        # Автоширина с разумным ограничением
        for column_cells in ws.columns:
            max_length = 0

            for cell in column_cells:
                value = "" if cell.value is None else str(cell.value)
                max_length = max(max_length, len(value))

            width = min(max(max_length + 2, 10), 45)
            ws.column_dimensions[column_cells[0].column_letter].width = width


print(f"Готово: {OUTPUT_FILE.resolve()}")
